The Goal of this notebook is to read all the Parquet files I parsed into, and determine just how much detectable error there is with a match threshold of 0.85. I should be able to find any "?" or any string that fails the validator

In [7]:
import os
from glob import glob
import pandas as pd
from allsky.validation import is_valid_date, is_valid_time, is_valid_exposure, is_valid_filename

files = sorted(glob(os.path.join(os.getcwd(), '../..' , 'data', '*.parquet')))

records = []
for file in files:
    df = pd.read_parquet(file)
    record = {'file': os.path.basename(file)}
    for col in df.columns:
        record[col + '_unknowns'] = df[col].str.contains('\\?').sum()
        record[col + '_schar'] = df[col].str.contains('s').sum()

        if col == 'date':
            invalids = ~df[col].apply(is_valid_date)
            record[col + '_invalid'] = invalids.sum()
        elif col == 'time':
            invalids = ~df[col].apply(is_valid_time)
            record[col + '_invalid'] = invalids.sum()
        elif col == 'exposure':
            invalids = ~df[col].apply(is_valid_exposure)
            record[col + '_invalid'] = invalids.sum()
        elif col == 'filename':
            invalids = ~df[col].apply(is_valid_filename)
            record[col + '_invalid'] = invalids.sum()
        else:
            record[col + '_invalid'] = None
    records.append(record)

errors_df = pd.DataFrame.from_records(records)
errors_df

,file,date_unknowns,date_schar,date_invalid,time_unknowns,time_schar,time_invalid,exposure_unknowns,exposure_schar,exposure_invalid,filename_unknowns,filename_schar,filename_invalid
0,2010-08.parquet,17,0,17,8,0,8,0,0,104,4,0,4
1,2010-09.parquet,0,0,0,0,0,0,0,0,0,0,0,0
2,2010-10.parquet,0,0,0,0,0,0,0,0,0,0,0,0
3,2010-11.parquet,0,0,0,0,0,0,0,0,0,3,0,3
4,2010-12.parquet,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,2020-08.parquet,15100,0,15100,15100,0,15100,15100,0,15100,15100,0,15100
95,2020-09.parquet,14656,0,14656,14655,0,14655,14654,0,14656,14656,0,14656
96,2020-10.parquet,15814,0,15814,15814,0,15814,15806,0,15814,15814,0,15814
97,2020-11.parquet,15386,0,15386,15386,0,15386,15386,0,15386,15386,0,15386


It can be clearly seen that the parse errors begin suddenly after 2015, when the camera changed or was replaced. At which point the text is antialiased and the existing parse fails